In [1]:
from mlflow.tracking import MlflowClient


In [2]:
from mlflow.entities import ViewType

In [3]:

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

2026/08/16 03:30:37 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/16 03:30:37 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


In [4]:
experiments = client.search_experiments()

for exp in experiments:
    print(exp.experiment_id, exp.name)

2 experiment-from-code
1 nyc-taxi-experiment
0 Default


In [5]:
client.create_experiment("experiment-from-code")

MlflowException: Experiment(name=experiment-from-code) already exists. Error: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.IntegrityError) UNIQUE constraint failed: experiments.name
[SQL: INSERT INTO experiments (name, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?)]
[parameters: ('experiment-from-code', None, 'active', 1786677811547, 1786677811547)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [6]:
runs = client.search_runs(experiment_ids=["1"], filter_string="metrics.rmse < 7.0", run_view_type=ViewType.ACTIVE_ONLY, max_results=5, order_by=["metrics.rmse ASC"])

In [7]:

for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: c885336c3094421db18013de380305f8, rmse: 6.3573
run id: 12562b70b2b04f36853832d97f62cd7a, rmse: 6.3698
run id: 469fdd9c56724e2691a43546d100a900, rmse: 6.3698
run id: 4a4da2c8b58449269552391ce83946d7, rmse: 6.3698
run id: bfd0795688534f44a22e4fb1fa0227bf, rmse: 6.3699


In [4]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [5]:
run_id = "12562b70b2b04f36853832d97f62cd7a"
model_uri = f"runs:/{run_id}/models_mlflow"

mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor")

2026/08/16 03:30:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/16 03:30:50 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/08/16 03:30:50 WARNING mlflow.tracking._model_registry.fluent: Run with id 12562b70b2b04f36853832d97f62cd7a has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-5a70a22536814f58a5bc9f0048870adf instead
Created version '3' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1786851050757, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1786851050757, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='12562b70b2b04f36853832d97f62cd7a', run_link=None, source='models:/m-5a70a22536814f58a5bc9f0048870adf', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [6]:
model_name = "nyc-taxi-regressor"
lastest_versions = client.get_latest_versions(name=model_name)

for version in lastest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}, run_id: {version.run_id}")  

version: 3, stage: None, run_id: 12562b70b2b04f36853832d97f62cd7a


/tmp/ipykernel_2354/2967156749.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  lastest_versions = client.get_latest_versions(name=model_name)


In [7]:
client.transition_model_version_stage(name=model_name, version=3, stage="Staging", archive_existing_versions=False)

/tmp/ipykernel_2354/3223653403.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(name=model_name, version=3, stage="Staging", archive_existing_versions=False)


<ModelVersion: aliases=[], creation_timestamp=1786851050757, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1786851286033, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='12562b70b2b04f36853832d97f62cd7a', run_link=None, source='models:/m-5a70a22536814f58a5bc9f0048870adf', status='READY', status_message=None, tags={}, user_id=None, version=3>